In [ ]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.multioutput import ClassifierChain
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import hamming_loss, f1_score, jaccard_score, accuracy_score
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def get_data():
    return None

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = get_data()

## 1. Binary Relevance
- **Problem transformation**: Decomposes multi-label problem into |L| independent binary classification problems
- For each label λ_j ∈ L, trains binary classifier h_j: **X** → {0,1}
- Final prediction: **y** = [h_1(**x**), h_2(**x**), ..., h_|L|(**x**)]
- **Assumption**: Labels are conditionally independent given features
- **Training complexity**: O(|L| × T_base) where T_base is base classifier training time
- **Inference complexity**: O(|L| × I_base) where I_base is base classifier inference time
- **Advantage**: Parallelizable, works with any binary classifier
- **Limitation**: Ignores label correlations and dependencies

In [ ]:
br_model = MultiOutputClassifier(LogisticRegression(random_state=42, max_iter=1000))

## 2. Classifier Chains
- **Sequential modeling**: Models label dependencies by chaining binary classifiers
- Defines ordering over labels: λ_1, λ_2, ..., λ_|L|
- For label λ_j, trains classifier h_j: **X** × {0,1}^(j-1) → {0,1}
- **Feature augmentation**: h_j uses original features **x** plus previous predictions [y_1, ..., y_(j-1)]
- **Training**: h_j trained on (**x**, y_1, ..., y_(j-1)) → y_j using ground truth labels
- **Inference**: Sequential prediction where h_j(**x**, ŷ_1, ..., ŷ_(j-1)) → ŷ_j
- **Training complexity**: O(|L| × T_base) - same as BR but features grow
- **Inference complexity**: O(|L| × I_base) - sequential, not parallelizable
- **Label ordering**: Can use random, frequency-based, or correlation-based ordering
- **Advantage**: Captures label dependencies explicitly
- **Limitation**: Error propagation through chain, sensitive to label ordering

In [ ]:
cc_model = ClassifierChain(LogisticRegression(random_state=42, max_iter=1000), 
                          random_state=42)

In [81]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
import json
import pandas as pd

In [82]:
path_dataset_train = "../../2025 EXIST/EXIST 2025 Tweets Dataset/training/EXIST2025_training.json"

In [83]:
def load_data(path):
    with open(path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
        
    df_raw = pd.DataFrame.from_dict(raw_data, orient="index").reset_index()
    df_raw = df_raw.rename(columns={"index": "id"})
    return df_raw

In [84]:
data_train = load_data(path_dataset_train)

In [85]:
path_hard_train1_3 = "../../2025 EXIST/evaluation/golds/EXIST2025_training_task1_3_gold_hard.json"

In [86]:
def load_eval(path):
    with open(path, 'r', encoding='utf-8') as f:
        data_from_file = json.load(f)

    df = pd.DataFrame(data_from_file)
    return df

In [87]:
classes_task1_3 = ['NO','IDEOLOGICAL-INEQUALITY', 'MISOGYNY-NON-SEXUAL-VIOLENCE',
                   'STEREOTYPING-DOMINANCE', 'OBJECTIFICATION','SEXUAL-VIOLENCE']

In [88]:
# Load soft and hard evaluations from disk
hard_train1_3 = load_eval(path_hard_train1_3)

# Rename target columns
hard_train1_3 = hard_train1_3.rename(columns={'value': 'hard_train1_3'})

# One-hot encode based on the string values in the column
list_train1_3_df = pd.DataFrame(hard_train1_3['hard_train1_3'].apply(
    lambda lst: {k: k in lst for k in classes_task1_3} if isinstance(lst, list) else {k: False for k in classes_task1_3}
    ).tolist())
hard_train1_3 = pd.concat([hard_train1_3,list_train1_3_df], axis=1)

# Append to dataset
data_train = pd.merge(data_train, hard_train1_3[['id','NO','IDEOLOGICAL-INEQUALITY','MISOGYNY-NON-SEXUAL-VIOLENCE','STEREOTYPING-DOMINANCE','OBJECTIFICATION','SEXUAL-VIOLENCE']], on='id', how='left')

In [89]:
data_train_en = data_train[data_train['lang']=='en']

In [90]:
data_train_small = data_train_en[['tweet', 'IDEOLOGICAL-INEQUALITY','MISOGYNY-NON-SEXUAL-VIOLENCE','STEREOTYPING-DOMINANCE','OBJECTIFICATION','SEXUAL-VIOLENCE']]

In [91]:
data_train_small.describe()

,tweet,IDEOLOGICAL-INEQUALITY,MISOGYNY-NON-SEXUAL-VIOLENCE,STEREOTYPING-DOMINANCE,OBJECTIFICATION,SEXUAL-VIOLENCE
count,3260,2864,2864,2864,2864,2864
unique,3260,2,2,2,2,2
top,FFS! How about laying the blame on the bastard...,False,False,False,False,False
freq,1,2383,2560,2251,2372,2548


In [92]:
df_train = data_train_small.copy()
df_train = df_train[(df_train['IDEOLOGICAL-INEQUALITY'].notna()) | (df_train['MISOGYNY-NON-SEXUAL-VIOLENCE'].notna()) | (df_train['STEREOTYPING-DOMINANCE'].notna()) | (df_train['OBJECTIFICATION'].notna()) | (df_train['SEXUAL-VIOLENCE'].notna())]
df_train = df_train[(df_train['IDEOLOGICAL-INEQUALITY']) | (df_train['MISOGYNY-NON-SEXUAL-VIOLENCE']) | (df_train['STEREOTYPING-DOMINANCE']) | (df_train['OBJECTIFICATION']) | (df_train['SEXUAL-VIOLENCE'])]  
df_train = df_train.rename(columns={'tweet': 'text'})

In [93]:
df_train[:10]

,text,IDEOLOGICAL-INEQUALITY,MISOGYNY-NON-SEXUAL-VIOLENCE,STEREOTYPING-DOMINANCE,OBJECTIFICATION,SEXUAL-VIOLENCE
3661,Writing a uni essay in my local pub with a cof...,False,False,True,True,False
3662,@UniversalORL it is 2021 not 1921. I dont appr...,False,False,True,True,False
3665,According to a customer I have plenty of time ...,False,False,True,True,False
3666,"So only 'blokes' drink beer? Sorry, but if you...",False,False,True,False,False
3670,#EverydaySexism means women usually end up in ...,True,False,True,False,False
3672,@orlamuldoon @NWCI @IrishRunnerMag @ReclaimTS ...,False,False,False,True,False
3674,@MarkPaulTimes @colettebrowne #EveryDaySexism ...,False,True,False,False,False
3675,@RMatthewsPsyEdu @ITV @jamesmartinchef @Everyd...,True,False,True,False,False
3676,@Geek_Pride @kathrynstimpson @medicalpoke @Eve...,True,False,False,False,False
3677,"February 28, 2022: Daily Mail website voyeuris...",False,True,True,False,False


In [ ]:
df_big = pd.concat([df.copy() for _ in range(10)], ignore_index=True)

NameError: name 'df' is not defined

In [ ]:
len(df_big)

NameError: name 'df_big' is not defined

In [94]:
path_dataset_eval = "../../2025 EXIST/EXIST 2025 Tweets Dataset/dev/EXIST2025_dev.json"

In [95]:
data_eval = load_data(path_dataset_eval)

In [96]:
path_hard_eval1_3 = "../../2025 EXIST/evaluation/golds/EXIST2025_dev_task1_3_gold_hard.json"

In [97]:
# Load soft and hard evaluations from disk
hard_eval1_3 = load_eval(path_hard_eval1_3)

# Rename target columns
hard_eval1_3 = hard_eval1_3.rename(columns={'value': 'hard_eval1_3'})

# One-hot encode based on the string values in the column
list_eval1_3_df = pd.DataFrame(hard_eval1_3['hard_eval1_3'].apply(
    lambda lst: {k: k in lst for k in classes_task1_3} if isinstance(lst, list) else {k: False for k in classes_task1_3}
    ).tolist())
hard_eval1_3 = pd.concat([hard_eval1_3,list_eval1_3_df], axis=1)

# Append to dataset
data_eval = pd.merge(data_eval, hard_eval1_3[['id','NO','IDEOLOGICAL-INEQUALITY','MISOGYNY-NON-SEXUAL-VIOLENCE','STEREOTYPING-DOMINANCE','OBJECTIFICATION','SEXUAL-VIOLENCE']], on='id', how='left')

In [98]:
data_eval_en = data_eval[data_eval['lang']=='en']

In [99]:
data_eval_small = data_eval_en[['tweet', 'IDEOLOGICAL-INEQUALITY','MISOGYNY-NON-SEXUAL-VIOLENCE','STEREOTYPING-DOMINANCE','OBJECTIFICATION','SEXUAL-VIOLENCE']]

In [100]:
data_eval_small.describe()

,tweet,IDEOLOGICAL-INEQUALITY,MISOGYNY-NON-SEXUAL-VIOLENCE,STEREOTYPING-DOMINANCE,OBJECTIFICATION,SEXUAL-VIOLENCE
count,489,444,444,444,444,444
unique,489,2,2,2,2,2
top,"@Mike_Fabricant “You should smile more, love. ...",False,False,False,False,False
freq,1,349,376,339,349,395


In [101]:
df_eval = data_eval_small.copy()
df_eval = df_eval[(df_eval['IDEOLOGICAL-INEQUALITY'].notna()) | (df_eval['MISOGYNY-NON-SEXUAL-VIOLENCE'].notna()) | (df_eval['STEREOTYPING-DOMINANCE'].notna()) | (df_eval['OBJECTIFICATION'].notna()) | (df_eval['SEXUAL-VIOLENCE'].notna())]
df_eval = df_eval[(df_eval['IDEOLOGICAL-INEQUALITY']) | (df_eval['MISOGYNY-NON-SEXUAL-VIOLENCE']) | (df_eval['STEREOTYPING-DOMINANCE']) | (df_eval['OBJECTIFICATION']) | (df_eval['SEXUAL-VIOLENCE'])]  
df_eval = df_eval.rename(columns={'tweet': 'text'})

In [102]:
df_eval[:10]

,text,IDEOLOGICAL-INEQUALITY,MISOGYNY-NON-SEXUAL-VIOLENCE,STEREOTYPING-DOMINANCE,OBJECTIFICATION,SEXUAL-VIOLENCE
550,@BBCWomansHour @LabWomenDec @EverydaySexism Sh...,True,False,False,True,False
551,#everydaysexism Some man moving my suitcase in...,False,False,True,False,False
556,@ReproRights @AbortionStories Getting Twitter ...,False,False,False,True,False
559,@esjayXX @EcuadorianMum @monsalore They so rem...,False,False,False,True,False
561,One of the depressing things about #NotAllMen ...,True,False,True,False,False
566,"Shit is crazy. If u want an open relationship,...",False,False,True,False,False
568,@GoldenSteeler06 @Matthew07219782 @PocketMaxim...,False,False,False,False,True
569,I sincerely wish the US was this progressive o...,True,True,True,True,True
577,#GOP raising Taxes and controlling #WomensRigh...,True,False,False,False,False
578,Freedom convoy Toronto waving Pro-Life flag.Th...,True,False,False,True,False


In [103]:
len(df_eval)

194

In [104]:
df_train.describe()

,text,IDEOLOGICAL-INEQUALITY,MISOGYNY-NON-SEXUAL-VIOLENCE,STEREOTYPING-DOMINANCE,OBJECTIFICATION,SEXUAL-VIOLENCE
count,1131,1131,1131,1131,1131,1131
unique,1131,2,2,2,2,2
top,Writing a uni essay in my local pub with a cof...,False,False,True,False,False
freq,1,650,827,613,639,815


In [ ]:
import pandas as pd
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.svm import SVC
from sklearn.multioutput import MultiOutputClassifier, ClassifierChain
import time
import contractions
from nltk.tokenize import TweetTokenizer

# Download required NLTK data
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')

# Initialize
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
tokenizer = TweetTokenizer()

def expand_contractions(texts: str) -> str:
    return contractions.fix(texts) if isinstance(texts, str) else texts

def reduce_repeated_characters(texts: str) -> str:
    repeat_pattern = re.compile(r'(.)\1{2,}')
    return repeat_pattern.sub(r'\1', texts)

# Negation handling 
def handle_negation(tokens):
    result = []
    negate = False
    for word in tokens:
        if word in ['not', 'no', 'never', "n't"]:
            negate = True
            continue
        if negate:
            result.append('not_' + word)
            negate = False
        else:
            result.append(word)
    return result

# Preprocessing
def preprocess_tweet(text):
    text = text.lower()
    text = expand_contractions(text)  # Expand contractions
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)  # Remove URLs
    text = re.sub(r"@\w+", 'USER', text)                 # Replace mentions
    #text = re.sub(r"@\w+", '', text)  # Remove mentions
    text = re.sub(r"#", '', text)                        # Keep hashtag word
    #text = re.sub(r"#\w+", '', text)  # Remove hashtags
    #text = re.sub(r"[^a-z\s]", '', text)                 # Remove non-letters
    text = reduce_repeated_characters(text)
    
    tokens = tokenizer.tokenize(text)
    tokens = handle_negation(tokens)
    tokens = [lemmatizer.lemmatize(word) for word in tokens 
              if word not in stop_words and len(word) > 1]
    
    return ' '.join(tokens)


# Convert one-hot label columns to a single categorical label


# Assuming df_train and df_eval have columns: 'text', 'label1', 'label2', ..., 'label5'
# Each label column should be binary (0/1)

# Example label columns:
label_cols = ['IDEOLOGICAL-INEQUALITY','MISOGYNY-NON-SEXUAL-VIOLENCE','STEREOTYPING-DOMINANCE','OBJECTIFICATION','SEXUAL-VIOLENCE']

df_tr = df_train.copy()
df_ev = df_eval.copy()

# Preprocess text
df_tr['clean_text'] = df_tr['text'].apply(preprocess_tweet)
df_ev['clean_text'] = df_ev['text'].apply(preprocess_tweet)

# Vectorize text
vectorizer = TfidfVectorizer(max_features=10000)
X_train = vectorizer.fit_transform(df_tr['clean_text'])
X_eval = vectorizer.transform(df_ev['clean_text'])

# Prepare multi-label targets
y_train = df_tr[label_cols].astype(int)
y_eval = df_ev[label_cols].astype(int)

# Define base classifier (you can choose LogisticRegression, SVC, etc.)
base_clf = LogisticRegression(max_iter=1000)

# Binary Relevance
br_model = MultiOutputClassifier(base_clf)
start = time.time()
br_model.fit(X_train, y_train)
print(f"Binary Relevance training took {time.time() - start:.2f}s")

y_pred_br = br_model.predict(X_eval)
print("Binary Relevance classification report:")
print(classification_report(y_eval, y_pred_br, zero_division=0))

# Classifier Chains
cc_model = ClassifierChain(base_clf)
start = time.time()
cc_model.fit(X_train, y_train)
print(f"Classifier Chains training took {time.time() - start:.2f}s")

y_pred_cc = cc_model.predict(X_eval)
print("Classifier Chains classification report:")
print(classification_report(y_eval, y_pred_cc, zero_division=0))


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/lucfaessler/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/lucfaessler/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/lucfaessler/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/lucfaessler/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Binary Relevance training took 0.03s
Binary Relevance classification report:
              precision    recall  f1-score   support

           0       0.71      0.49      0.58        95
           1       0.60      0.04      0.08        68
           2       0.66      0.82      0.73       105
           3       0.78      0.38      0.51        95
           4       0.86      0.12      0.21        49

   micro avg       0.70      0.43      0.53       412
   macro avg       0.72      0.37      0.42       412
weighted avg       0.71      0.43      0.48       412
 samples avg       0.61      0.48      0.50       412

Classifier Chains training took 0.05s
Classifier Chains classification report:
              precision    recall  f1-score   support

           0       0.71      0.49      0.58        95
           1       0.67      0.06      0.11        68
           2       0.65      0.77      0.71       105
           3       0.64      0.64      0.64        95
           4       0.55      0

In [109]:
def get_labels(row):
    return [label for label in label_cols if row[label] == 1]

df_ev['labels'] = df_ev.apply(get_labels, axis=1)

from collections import Counter
label_counts = Counter(label for labels in df_ev['labels'] for label in labels)
print("Label distribution across all samples:")
for label, count in label_counts.items():
    print(f"{label}: {count} samples")

Label distribution across all samples:
IDEOLOGICAL-INEQUALITY: 95 samples
OBJECTIFICATION: 95 samples
STEREOTYPING-DOMINANCE: 105 samples
SEXUAL-VIOLENCE: 49 samples
MISOGYNY-NON-SEXUAL-VIOLENCE: 68 samples
